<a href="https://colab.research.google.com/github/Dorthi12/SIH-26/blob/main/03_production_risk_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌾 Model 3 — Crop Production Risk & Failure Prediction

## Smart Agriculture Decision-Support System

---

## 1. Model Objective

Model 3 predicts the probability that a selected:

- State
- District
- Crop
- Season
- Year
- Cultivated Area

combination will result in **zero agricultural production**.

The model acts as a **production-risk / failure-risk predictor**.

Instead of only recommending a crop based on historical performance, the system will also estimate whether that crop has a significant historical risk of producing zero output.

### Example

A farmer requests:

```text
State    : Maharashtra
District : Pune
Crop     : Rice
Season   : Kharif
Area     : 5 hectares
Year     : 2025

                         FARMER INPUT
                              │
                              ▼
                ┌─────────────────────────┐
                │   Location / Crop Data  │
                │ State / District / Crop │
                │ Season / Area / Year    │
                └────────────┬────────────┘
                             │
          ┌──────────────────┼──────────────────┐
          │                  │                  │
          ▼                  ▼                  ▼
     MODEL 1             MODEL 2            MODEL 3
  Yield Prediction    Crop Recommendation   Risk Prediction
          │                  │                  │
          ▼                  ▼                  ▼
   Expected Yield       Top-K Crops       Failure Probability
          │                  │                  │
          └──────────────────┼──────────────────┘
                             ▼
                  FINAL DECISION ENGINE
                             │
                             ▼
                  Agentic Recommendation

MODEL 1
"What yield can we expect?"
        ↓
Predicted yield


MODEL 2
"Which crops are historically suitable?"
        ↓
Top-K crops


MODEL 3
"How risky / reliable is this crop choice?"
        ↓
Risk / stability score

DATA REQUIRED:-

state

district

crop

season

area

crop_year

In [ ]:
# ============================================================
# MODEL 3 — PRODUCTION FAILURE / ZERO-PRODUCTION RISK
# STEP 1 — LOAD DATA
# ============================================================

import pandas as pd
import numpy as np
import os

DATA_PATH = "/content/apy_clean.csv"

apy = pd.read_csv(DATA_PATH)

print("=" * 60)
print("MODEL 3 — DATA LOADED")
print("=" * 60)

print("Shape:", apy.shape)
print("\nColumns:")
print(apy.columns.tolist())

display(apy.head())

MODEL 3 — DATA LOADED
Shape: (344727, 9)

Columns:
['state', 'district', 'crop', 'crop_year', 'season', 'area', 'production', 'yield', 'zero_production_flag']


,state,district,crop,crop_year,season,area,production,yield,zero_production_flag
0,Andaman and Nicobar Island,NICOBARS,Arecanut,2000,Kharif,1254.0,2000.0,1.594896,0
1,Andaman and Nicobar Island,NICOBARS,Arecanut,2001,Kharif,1254.0,2061.0,1.643541,0
2,Andaman and Nicobar Island,NICOBARS,Arecanut,2002,Whole Year,1258.0,2083.0,1.655803,0
3,Andaman and Nicobar Island,NICOBARS,Arecanut,2003,Whole Year,1261.0,1525.0,1.209358,0
4,Andaman and Nicobar Island,NICOBARS,Arecanut,2004,Whole Year,1264.7,806.0,0.637305,0


In [ ]:
# ============================================================
# STEP 2 — TARGET VALIDATION
# ============================================================

print("=" * 60)
print("TARGET VALIDATION")
print("=" * 60)

print("\nZero-production flag distribution:")
print(apy["zero_production_flag"].value_counts())

print("\nZero-production percentage:")
print(
    apy["zero_production_flag"].mean() * 100
)

print("\nMissing values:")
print(
    apy[
        [
            "state",
            "district",
            "crop",
            "season",
            "area",
            "zero_production_flag"
        ]
    ].isna().sum()
)

TARGET VALIDATION

Zero-production flag distribution:
zero_production_flag
0    338334
1      6393
Name: count, dtype: int64

Zero-production percentage:
1.854510960847279

Missing values:
state                   0
district                0
crop                    0
season                  0
area                    0
zero_production_flag    0
dtype: int64


In [ ]:
# ============================================================
# STEP 3 — DEFINE FEATURES
# ============================================================

MODEL_3_FEATURES = [
    "state",
    "district",
    "crop",
    "season",
    "area"
]

TARGET = "zero_production_flag"

X = apy[MODEL_3_FEATURES].copy()
y = apy[TARGET].copy()

print("=" * 60)
print("MODEL 3 — FEATURE CONFIGURATION")
print("=" * 60)

print("\nFeatures:")
for feature in MODEL_3_FEATURES:
    print("✓", feature)

print("\nTarget:")
print("✓", TARGET)

print("\nForbidden leakage columns:")
print("✗ production")
print("✗ yield")

print("\nIntentionally excluded:")
print("✗ crop_year")

MODEL 3 — FEATURE CONFIGURATION

Features:
✓ state
✓ district
✓ crop
✓ season
✓ area

Target:
✓ zero_production_flag

Forbidden leakage columns:
✗ production
✗ yield

Intentionally excluded:
✗ crop_year


In [ ]:
# ============================================================
# STEP 4 — FUTURE YEAR COMPATIBILITY CHECK
# ============================================================

print("=" * 60)
print("FUTURE YEAR COMPATIBILITY")
print("=" * 60)

print("""
Model 3 does NOT use crop_year as a model feature.

Therefore the model learns historical production-failure
patterns from:

    state
    district
    crop
    season
    area

The requested prediction year is handled by the application
layer rather than being required to exist in the training data.

Example:

    Requested year = 2026

The model can still evaluate:

    Maharashtra
    Pune
    Rice
    Kharif
    5 hectares

using historical evidence.
""")

FUTURE YEAR COMPATIBILITY

Model 3 does NOT use crop_year as a model feature.

Therefore the model learns historical production-failure
patterns from:

    state
    district
    crop
    season
    area

The requested prediction year is handled by the application
layer rather than being required to exist in the training data.

Example:

    Requested year = 2026

The model can still evaluate:

    Maharashtra
    Pune
    Rice
    Kharif
    5 hectares

using historical evidence.



In [ ]:
# ============================================================
# STEP 5 — CARDINALITY CHECK
# ============================================================

print("=" * 60)
print("FEATURE CARDINALITY")
print("=" * 60)

for col in MODEL_3_FEATURES:
    print(f"{col:15s}: {apy[col].nunique():5d} unique values")

FEATURE CARDINALITY
state          :    37 unique values
district       :   707 unique values
crop           :    55 unique values
season         :     6 unique values
area           : 47714 unique values


In [ ]:
# ============================================================
# STEP 6 — TEMPORAL TRAIN / VALIDATION / TEST SPLIT
# ============================================================

print("=" * 60)
print("MODEL 3 — TEMPORAL SPLIT")
print("=" * 60)

# Make sure year is numeric
apy["crop_year"] = pd.to_numeric(
    apy["crop_year"],
    errors="coerce"
)

# Remove rows where year is unavailable
apy_model = apy.dropna(
    subset=["crop_year"]
).copy()

apy_model["crop_year"] = (
    apy_model["crop_year"]
    .astype(int)
)

# ------------------------------------------------------------
# Define periods
# ------------------------------------------------------------

train_data = apy_model[
    apy_model["crop_year"] <= 2017
].copy()

val_data = apy_model[
    apy_model["crop_year"] == 2018
].copy()

test_data = apy_model[
    apy_model["crop_year"] == 2019
].copy()

excluded_data = apy_model[
    apy_model["crop_year"] >= 2020
].copy()

# ------------------------------------------------------------
# Build X / y
# ------------------------------------------------------------

X_train = train_data[MODEL_3_FEATURES]
y_train = train_data[TARGET]

X_val = val_data[MODEL_3_FEATURES]
y_val = val_data[TARGET]

X_test = test_data[MODEL_3_FEATURES]
y_test = test_data[TARGET]

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\nTraining:")
print(
    f"Years: {train_data['crop_year'].min()} - "
    f"{train_data['crop_year'].max()}"
)
print(f"Rows: {len(train_data):,}")

print("\nValidation:")
print(
    f"Year: {sorted(val_data['crop_year'].unique())}"
)
print(f"Rows: {len(val_data):,}")

print("\nTest:")
print(
    f"Year: {sorted(test_data['crop_year'].unique())}"
)
print(f"Rows: {len(test_data):,}")

print("\nExcluded:")
print(
    f"Years: {sorted(excluded_data['crop_year'].unique())}"
)
print(f"Rows: {len(excluded_data):,}")

MODEL 3 — TEMPORAL SPLIT

Training:
Years: 1997 - 2017
Rows: 306,892

Validation:
Year: [np.int64(2018)]
Rows: 18,281

Test:
Year: [np.int64(2019)]
Rows: 19,246

Excluded:
Years: [np.int64(2020)]
Rows: 308


In [ ]:
# ============================================================
# STEP 7 — CLASS DISTRIBUTION BY SPLIT
# ============================================================

print("=" * 60)
print("CLASS DISTRIBUTION")
print("=" * 60)

def show_class_distribution(name, y):

    counts = y.value_counts().sort_index()
    percentages = (
        y.value_counts(normalize=True)
        .sort_index()
        .mul(100)
    )

    result = pd.DataFrame({
        "count": counts,
        "percentage": percentages
    })

    print(f"\n{name}")
    display(result)


show_class_distribution(
    "TRAINING",
    y_train
)

show_class_distribution(
    "VALIDATION",
    y_val
)

show_class_distribution(
    "TEST",
    y_test
)

CLASS DISTRIBUTION

TRAINING


,count,percentage
zero_production_flag,,
0,301083,98.107152
1,5809,1.892848



VALIDATION


,count,percentage
zero_production_flag,,
0,17990,98.408183
1,291,1.591817



TEST


,count,percentage
zero_production_flag,,
0,18953,98.477606
1,293,1.522394


In [ ]:
# ============================================================
# STEP 8 — CLASS IMBALANCE WEIGHT
# ============================================================

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

class_weight_ratio = negative_count / positive_count

print("=" * 60)
print("CLASS IMBALANCE")
print("=" * 60)

print(f"Negative class (0): {negative_count:,}")
print(f"Positive class (1): {positive_count:,}")

print(f"\nNegative / Positive ratio: {class_weight_ratio:.4f}")

print("\nWe will use this ratio as the positive-class weight.")

CLASS IMBALANCE
Negative class (0): 301,083
Positive class (1): 5,809

Negative / Positive ratio: 51.8304

We will use this ratio as the positive-class weight.


In [ ]:
# ============================================================
# STEP 9 — IMPORT MODEL + METRICS
# ============================================================

!pip -q install catboost

from catboost import CatBoostClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

print("Libraries ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00
Libraries ready.


In [ ]:
# ============================================================
# STEP 10 — MODEL 3 CATBOOST TRAINING
# ============================================================

CAT_FEATURES = [
    "state",
    "district",
    "crop",
    "season"
]

cat_feature_indices = [
    MODEL_3_FEATURES.index(col)
    for col in CAT_FEATURES
]

model_3 = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

model_3.fit(
    X_train,
    y_train,

    cat_features=cat_feature_indices,

    eval_set=(X_val, y_val),

    use_best_model=True,
    early_stopping_rounds=100
)

print("\n" + "=" * 60)
print("MODEL 3 TRAINING COMPLETED")
print("=" * 60)

print("Best iteration:", model_3.get_best_iteration())
print("Best validation score:", model_3.get_best_score())

0:	test: 0.9078451	best: 0.9078451 (0)	total: 371ms	remaining: 6m 10s
100:	test: 0.9430898	best: 0.9446351 (75)	total: 38s	remaining: 5m 38s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9446351448
bestIteration = 75

Shrink model to first 76 iterations.

MODEL 3 TRAINING COMPLETED
Best iteration: 75
Best validation score: {'learn': {'Logloss': 0.1721019474179955}, 'validation': {'Logloss': 0.30859668592981054, 'AUC': 0.9446351447635093}}


In [ ]:
# ============================================================
# STEP 11 — PREDICT PROBABILITIES
# ============================================================

val_prob = model_3.predict_proba(X_val)[:, 1]
test_prob = model_3.predict_proba(X_test)[:, 1]

print("Validation predictions:", len(val_prob))
print("Test predictions:", len(test_prob))

print("\nValidation probability statistics:")
print(pd.Series(val_prob).describe())

print("\nTest probability statistics:")
print(pd.Series(test_prob).describe())

Validation predictions: 18281
Test predictions: 19246

Validation probability statistics:
count    18281.000000
mean         0.152999
std          0.224203
min          0.004173
25%          0.028346
50%          0.065067
75%          0.146125
max          0.999336
dtype: float64

Test probability statistics:
count    19246.000000
mean         0.159788
std          0.228555
min          0.003704
25%          0.029700
50%          0.069187
75%          0.159727
max          0.999336
dtype: float64


In [ ]:
# ============================================================
# STEP 12 — THRESHOLD ANALYSIS
# ============================================================

thresholds = np.arange(0.05, 0.56, 0.05)

threshold_results = []

for threshold in thresholds:

    val_pred = (
        val_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        val_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        val_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        val_pred,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results = pd.DataFrame(
    threshold_results
)

display(
    threshold_results.round(4)
)

,threshold,precision,recall,f1
0,0.05,0.0270,0.9897,0.0526
1,0.10,0.0464,0.9794,0.0886
2,0.15,0.0620,0.9553,0.1164
3,0.20,0.0767,0.9347,0.1417
4,0.25,0.0858,0.8866,0.1565
5,0.30,0.0967,0.8729,0.1742
6,0.35,0.1076,0.8625,0.1913
7,0.40,0.1173,0.8488,0.2062
8,0.45,0.1275,0.8385,0.2214
9,0.50,0.1404,0.8282,0.2400


In [ ]:
# ============================================================
# STEP 13 — BEST VALIDATION THRESHOLD
# ============================================================

best_threshold_row = (
    threshold_results
    .sort_values("f1", ascending=False)
    .iloc[0]
)

BEST_THRESHOLD = float(
    best_threshold_row["threshold"]
)

print("=" * 60)
print("BEST VALIDATION THRESHOLD")
print("=" * 60)

print(f"Threshold : {BEST_THRESHOLD:.2f}")
print(f"Precision : {best_threshold_row['precision']:.4f}")
print(f"Recall    : {best_threshold_row['recall']:.4f}")
print(f"F1        : {best_threshold_row['f1']:.4f}")

BEST VALIDATION THRESHOLD
Threshold : 0.55
Precision : 0.1513
Recall    : 0.8144
F1        : 0.2553


In [ ]:
# ============================================================
# STEP 14 — FINAL TEST EVALUATION
# ============================================================

test_pred = (
    test_prob >= BEST_THRESHOLD
).astype(int)

test_precision = precision_score(
    y_test,
    test_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    test_pred,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    test_pred,
    zero_division=0
)

test_roc_auc = roc_auc_score(
    y_test,
    test_prob
)

test_pr_auc = average_precision_score(
    y_test,
    test_prob
)

print("=" * 60)
print("MODEL 3 — FINAL TEST METRICS")
print("=" * 60)

print(f"Threshold : {BEST_THRESHOLD:.2f}")
print(f"Precision : {test_precision:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"F1        : {test_f1:.4f}")
print(f"ROC-AUC   : {test_roc_auc:.4f}")
print(f"PR-AUC    : {test_pr_auc:.4f}")

MODEL 3 — FINAL TEST METRICS
Threshold : 0.55
Precision : 0.1294
Recall    : 0.7850
F1        : 0.2222
ROC-AUC   : 0.9077
PR-AUC    : 0.2141


In [ ]:
# ============================================================
# STEP 14 — FINAL TEST EVALUATION
# ============================================================

test_pred = (
    test_prob >= BEST_THRESHOLD
).astype(int)

test_precision = precision_score(
    y_test,
    test_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    test_pred,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    test_pred,
    zero_division=0
)

test_roc_auc = roc_auc_score(
    y_test,
    test_prob
)

test_pr_auc = average_precision_score(
    y_test,
    test_prob
)

print("=" * 60)
print("MODEL 3 — FINAL TEST METRICS")
print("=" * 60)

print(f"Threshold : {BEST_THRESHOLD:.2f}")
print(f"Precision : {test_precision:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"F1        : {test_f1:.4f}")
print(f"ROC-AUC   : {test_roc_auc:.4f}")
print(f"PR-AUC    : {test_pr_auc:.4f}")

MODEL 3 — FINAL TEST METRICS
Threshold : 0.55
Precision : 0.1294
Recall    : 0.7850
F1        : 0.2222
ROC-AUC   : 0.9077
PR-AUC    : 0.2141


In [ ]:
# ============================================================
# STEP 15 — CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    test_pred
)

print("=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

print("""
                 Predicted
                 0       1
Actual  0       TN      FP
        1       FN      TP
""")

print(cm)

CONFUSION MATRIX

                 Predicted
                 0       1
Actual  0       TN      FP
        1       FN      TP

[[17406  1547]
 [   63   230]]


In [ ]:
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test,
        test_pred,
        target_names=[
            "Normal production",
            "Zero production"
        ],
        zero_division=0
    )
)

CLASSIFICATION REPORT
                   precision    recall  f1-score   support

Normal production       1.00      0.92      0.96     18953
  Zero production       0.13      0.78      0.22       293

         accuracy                           0.92     19246
        macro avg       0.56      0.85      0.59     19246
     weighted avg       0.98      0.92      0.94     19246



In [ ]:
# ============================================================
# MODEL 3 V2 — HISTORICAL RISK FEATURE ENGINEERING
# ============================================================

print("=" * 60)
print("MODEL 3 V2 — PREPARING HISTORICAL RISK FEATURES")
print("=" * 60)

print("Dataset shape:", apy.shape)
print("Columns:")
print(apy.columns.tolist())

MODEL 3 V2 — PREPARING HISTORICAL RISK FEATURES
Dataset shape: (344727, 9)
Columns:
['state', 'district', 'crop', 'crop_year', 'season', 'area', 'production', 'yield', 'zero_production_flag']


In [ ]:
# ------------------------------------------------------------
# BASIC SANITY CHECK
# ------------------------------------------------------------

required_cols = [
    "state",
    "district",
    "crop",
    "crop_year",
    "season",
    "area",
    "production",
    "yield",
    "zero_production_flag"
]

missing_cols = [
    col for col in required_cols
    if col not in apy.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )

print("✓ All required columns present")

✓ All required columns present


In [ ]:
# ============================================================
# MODEL 3 V2 — TEMPORAL DATA SPLIT
# ============================================================

TRAIN_END_YEAR = 2017
VALIDATION_YEAR = 2018
TEST_YEAR = 2019

train_raw = apy[
    apy["crop_year"] <= TRAIN_END_YEAR
].copy()

val_raw = apy[
    apy["crop_year"] == VALIDATION_YEAR
].copy()

test_raw = apy[
    apy["crop_year"] == TEST_YEAR
].copy()

# 2020 is excluded from evaluation
excluded_raw = apy[
    apy["crop_year"] > TEST_YEAR
].copy()

print("=" * 60)
print("MODEL 3 V2 — TEMPORAL SPLIT")
print("=" * 60)

print(f"Training   : {train_raw['crop_year'].min()} - "
      f"{train_raw['crop_year'].max()} | "
      f"{len(train_raw):,} rows")

print(f"Validation : {VALIDATION_YEAR} | "
      f"{len(val_raw):,} rows")

print(f"Test       : {TEST_YEAR} | "
      f"{len(test_raw):,} rows")

print(f"Excluded   : {len(excluded_raw):,} rows")

print("\nZero-production distribution:")

print("\nTRAIN")
print(train_raw["zero_production_flag"].value_counts())

print("\nVALIDATION")
print(val_raw["zero_production_flag"].value_counts())

print("\nTEST")
print(test_raw["zero_production_flag"].value_counts())

MODEL 3 V2 — TEMPORAL SPLIT
Training   : 1997 - 2017 | 306,892 rows
Validation : 2018 | 18,281 rows
Test       : 2019 | 19,246 rows
Excluded   : 308 rows

Zero-production distribution:

TRAIN
zero_production_flag
0    301083
1      5809
Name: count, dtype: int64

VALIDATION
zero_production_flag
0    17990
1      291
Name: count, dtype: int64

TEST
zero_production_flag
0    18953
1      293
Name: count, dtype: int64


In [ ]:
# ============================================================
# MODEL 3 V2 — HISTORICAL FEATURE BUILDER
# ============================================================

import numpy as np
import pandas as pd


def build_historical_features(current_data, historical_data):
    """
    Build production-risk features using historical data only.

    current_data:
        Rows for which predictions are required.

    historical_data:
        Data strictly preceding current_data temporally.
    """

    current = current_data.copy()
    history = historical_data.copy()

    # --------------------------------------------------------
    # BASIC NUMERIC FEATURES
    # --------------------------------------------------------

    current["log_area"] = np.log1p(
        current["area"].clip(lower=0)
    )

    # --------------------------------------------------------
    # GLOBAL HISTORICAL RISK
    # --------------------------------------------------------

    global_zero_rate = (
        history["zero_production_flag"].mean()
    )

    current["historical_zero_rate_global"] = global_zero_rate

    # --------------------------------------------------------
    # CROP-LEVEL RISK
    # --------------------------------------------------------

    crop_zero = (
        history
        .groupby("crop")["zero_production_flag"]
        .mean()
        .rename("historical_crop_zero_rate")
    )

    current = current.merge(
        crop_zero,
        on="crop",
        how="left"
    )

    # --------------------------------------------------------
    # STATE-LEVEL RISK
    # --------------------------------------------------------

    state_zero = (
        history
        .groupby("state")["zero_production_flag"]
        .mean()
        .rename("historical_state_zero_rate")
    )

    current = current.merge(
        state_zero,
        on="state",
        how="left"
    )

    # --------------------------------------------------------
    # DISTRICT-LEVEL RISK
    # --------------------------------------------------------

    district_zero = (
        history
        .groupby("district")["zero_production_flag"]
        .mean()
        .rename("historical_district_zero_rate")
    )

    current = current.merge(
        district_zero,
        on="district",
        how="left"
    )

    # --------------------------------------------------------
    # STATE + CROP RISK
    # --------------------------------------------------------

    state_crop_zero = (
        history
        .groupby(["state", "crop"])["zero_production_flag"]
        .mean()
        .rename("historical_state_crop_zero_rate")
    )

    current = current.merge(
        state_crop_zero,
        on=["state", "crop"],
        how="left"
    )

    # --------------------------------------------------------
    # DISTRICT + CROP RISK
    # --------------------------------------------------------

    district_crop_zero = (
        history
        .groupby(["district", "crop"])["zero_production_flag"]
        .mean()
        .rename("historical_district_crop_zero_rate")
    )

    current = current.merge(
        district_crop_zero,
        on=["district", "crop"],
        how="left"
    )

    # --------------------------------------------------------
    # CROP + SEASON RISK
    # --------------------------------------------------------

    crop_season_zero = (
        history
        .groupby(["crop", "season"])["zero_production_flag"]
        .mean()
        .rename("historical_crop_season_zero_rate")
    )

    current = current.merge(
        crop_season_zero,
        on=["crop", "season"],
        how="left"
    )

    # --------------------------------------------------------
    # DISTRICT + CROP + SEASON RISK
    # --------------------------------------------------------

    district_crop_season_zero = (
        history
        .groupby(
            ["district", "crop", "season"]
        )["zero_production_flag"]
        .mean()
        .rename(
            "historical_district_crop_season_zero_rate"
        )
    )

    current = current.merge(
        district_crop_season_zero,
        on=["district", "crop", "season"],
        how="left"
    )

    # --------------------------------------------------------
    # HISTORICAL YIELD FEATURES
    # --------------------------------------------------------

    crop_yield = (
        history
        .groupby("crop")["yield"]
        .agg(
            historical_crop_mean_yield="mean",
            historical_crop_median_yield="median",
            historical_crop_std_yield="std",
            historical_crop_min_yield="min",
            historical_crop_max_yield="max"
        )
        .reset_index()
    )

    current = current.merge(
        crop_yield,
        on="crop",
        how="left"
    )

    # --------------------------------------------------------
    # HISTORICAL AREA FEATURES
    # --------------------------------------------------------

    crop_area = (
        history
        .groupby("crop")["area"]
        .agg(
            historical_crop_mean_area="mean",
            historical_crop_median_area="median"
        )
        .reset_index()
    )

    current = current.merge(
        crop_area,
        on="crop",
        how="left"
    )

    # --------------------------------------------------------
    # AREA RELATIVE TO HISTORICAL CROP AREA
    # --------------------------------------------------------

    current["area_vs_historical_crop_mean"] = (
        current["area"] /
        (current["historical_crop_mean_area"] + 1e-6)
    )

    # --------------------------------------------------------
    # FILL UNSEEN-CATEGORY VALUES
    # --------------------------------------------------------

    rate_columns = [
        col for col in current.columns
        if "zero_rate" in col
    ]

    for col in rate_columns:
        current[col] = current[col].fillna(
            global_zero_rate
        )

    numeric_columns = current.select_dtypes(
        include=[np.number]
    ).columns

    for col in numeric_columns:
        current[col] = current[col].replace(
            [np.inf, -np.inf],
            np.nan
        )

    return current

In [ ]:
# ============================================================
# MODEL 3 V2 — BUILD TRAINING FEATURES
# ============================================================

train_features = build_historical_features(
    current_data=train_raw,
    historical_data=train_raw
)

print("=" * 60)
print("TRAINING FEATURES CREATED")
print("=" * 60)

print("Shape:", train_features.shape)

print("\nNew features:")

historical_features = [
    col for col in train_features.columns
    if (
        "historical_" in col
        or col in [
            "log_area",
            "area_vs_historical_crop_mean"
        ]
    )
]

print(historical_features)

TRAINING FEATURES CREATED
Shape: (306892, 26)

New features:
['log_area', 'historical_zero_rate_global', 'historical_crop_zero_rate', 'historical_state_zero_rate', 'historical_district_zero_rate', 'historical_state_crop_zero_rate', 'historical_district_crop_zero_rate', 'historical_crop_season_zero_rate', 'historical_district_crop_season_zero_rate', 'historical_crop_mean_yield', 'historical_crop_median_yield', 'historical_crop_std_yield', 'historical_crop_min_yield', 'historical_crop_max_yield', 'historical_crop_mean_area', 'historical_crop_median_area', 'area_vs_historical_crop_mean']


In [ ]:
# ============================================================
# MODEL 3 V2 — LEAKAGE-SAFE EXPANDING HISTORICAL FEATURES
# ============================================================

def build_expanding_features(df):
    """
    Build historical features using only information available
    BEFORE each observation's year.

    This prevents temporal leakage.
    """

    df = df.copy().sort_values("crop_year").reset_index(drop=True)

    feature_rows = []

    years = sorted(df["crop_year"].unique())

    for year in years:

        current = df[df["crop_year"] == year].copy()
        history = df[df["crop_year"] < year].copy()

        # No historical information exists for the first year
        if len(history) == 0:
            current["historical_zero_rate_global"] = np.nan
            current["historical_crop_zero_rate"] = np.nan
            current["historical_state_zero_rate"] = np.nan
            current["historical_district_zero_rate"] = np.nan
            current["historical_state_crop_zero_rate"] = np.nan
            current["historical_district_crop_zero_rate"] = np.nan
            current["historical_crop_season_zero_rate"] = np.nan
            current["historical_district_crop_season_zero_rate"] = np.nan

            current["historical_crop_mean_yield"] = np.nan
            current["historical_crop_median_yield"] = np.nan
            current["historical_crop_std_yield"] = np.nan
            current["historical_crop_min_yield"] = np.nan
            current["historical_crop_max_yield"] = np.nan

            current["historical_crop_mean_area"] = np.nan
            current["historical_crop_median_area"] = np.nan

        else:

            global_rate = history[
                "zero_production_flag"
            ].mean()

            current["historical_zero_rate_global"] = global_rate

            # ------------------------------------------------
            # CROP
            # ------------------------------------------------

            crop_rate = (
                history
                .groupby("crop")["zero_production_flag"]
                .mean()
            )

            current["historical_crop_zero_rate"] = (
                current["crop"].map(crop_rate)
            )

            # ------------------------------------------------
            # STATE
            # ------------------------------------------------

            state_rate = (
                history
                .groupby("state")["zero_production_flag"]
                .mean()
            )

            current["historical_state_zero_rate"] = (
                current["state"].map(state_rate)
            )

            # ------------------------------------------------
            # DISTRICT
            # ------------------------------------------------

            district_rate = (
                history
                .groupby("district")["zero_production_flag"]
                .mean()
            )

            current["historical_district_zero_rate"] = (
                current["district"].map(district_rate)
            )

            # ------------------------------------------------
            # STATE + CROP
            # ------------------------------------------------

            state_crop_rate = (
                history
                .groupby(
                    ["state", "crop"]
                )["zero_production_flag"]
                .mean()
            )

            current["historical_state_crop_zero_rate"] = [
                state_crop_rate.get((s, c), np.nan)
                for s, c in zip(
                    current["state"],
                    current["crop"]
                )
            ]

            # ------------------------------------------------
            # DISTRICT + CROP
            # ------------------------------------------------

            district_crop_rate = (
                history
                .groupby(
                    ["district", "crop"]
                )["zero_production_flag"]
                .mean()
            )

            current["historical_district_crop_zero_rate"] = [
                district_crop_rate.get((d, c), np.nan)
                for d, c in zip(
                    current["district"],
                    current["crop"]
                )
            ]

            # ------------------------------------------------
            # CROP + SEASON
            # ------------------------------------------------

            crop_season_rate = (
                history
                .groupby(
                    ["crop", "season"]
                )["zero_production_flag"]
                .mean()
            )

            current["historical_crop_season_zero_rate"] = [
                crop_season_rate.get((c, s), np.nan)
                for c, s in zip(
                    current["crop"],
                    current["season"]
                )
            ]

            # ------------------------------------------------
            # DISTRICT + CROP + SEASON
            # ------------------------------------------------

            dcs_rate = (
                history
                .groupby(
                    ["district", "crop", "season"]
                )["zero_production_flag"]
                .mean()
            )

            current[
                "historical_district_crop_season_zero_rate"
            ] = [
                dcs_rate.get((d, c, s), np.nan)
                for d, c, s in zip(
                    current["district"],
                    current["crop"],
                    current["season"]
                )
            ]

            # ------------------------------------------------
            # HISTORICAL CROP YIELD
            # ------------------------------------------------

            crop_yield = (
                history
                .groupby("crop")["yield"]
                .agg([
                    "mean",
                    "median",
                    "std",
                    "min",
                    "max"
                ])
            )

            current["historical_crop_mean_yield"] = (
                current["crop"].map(crop_yield["mean"])
            )

            current["historical_crop_median_yield"] = (
                current["crop"].map(crop_yield["median"])
            )

            current["historical_crop_std_yield"] = (
                current["crop"].map(crop_yield["std"])
            )

            current["historical_crop_min_yield"] = (
                current["crop"].map(crop_yield["min"])
            )

            current["historical_crop_max_yield"] = (
                current["crop"].map(crop_yield["max"])
            )

            # ------------------------------------------------
            # HISTORICAL CROP AREA
            # ------------------------------------------------

            crop_area = (
                history
                .groupby("crop")["area"]
                .agg(["mean", "median"])
            )

            current["historical_crop_mean_area"] = (
                current["crop"].map(crop_area["mean"])
            )

            current["historical_crop_median_area"] = (
                current["crop"].map(crop_area["median"])
            )

        # ----------------------------------------------------
        # CURRENT-ROW FEATURES
        # ----------------------------------------------------

        current["log_area"] = np.log1p(
            current["area"].clip(lower=0)
        )

        current["area_vs_historical_crop_mean"] = (
            current["area"] /
            (current["historical_crop_mean_area"] + 1e-6)
        )

        feature_rows.append(current)

    result = pd.concat(
        feature_rows,
        ignore_index=True
    )

    return result

In [ ]:
# ============================================================
# BUILD LEAKAGE-SAFE TRAINING FEATURES
# ============================================================

train_expanding = build_expanding_features(train_raw)

print("=" * 60)
print("EXPANDING FEATURES CREATED")
print("=" * 60)

print("Shape:", train_expanding.shape)

print("\nYear range:")
print(
    train_expanding["crop_year"].min(),
    "→",
    train_expanding["crop_year"].max()
)

print("\nHistorical feature missingness:")
display(
    train_expanding[
        [
            "historical_zero_rate_global",
            "historical_crop_zero_rate",
            "historical_state_zero_rate",
            "historical_district_zero_rate",
            "historical_state_crop_zero_rate",
            "historical_district_crop_zero_rate",
            "historical_crop_season_zero_rate",
            "historical_district_crop_season_zero_rate"
        ]
    ].isna().sum()
)

EXPANDING FEATURES CREATED
Shape: (306892, 26)

Year range:
1997 → 2017

Historical feature missingness:


,0
historical_zero_rate_global,8529
historical_crop_zero_rate,8662
historical_state_zero_rate,10942
historical_district_zero_rate,12530
historical_state_crop_zero_rate,18141
historical_district_crop_zero_rate,23424
historical_crop_season_zero_rate,9930
historical_district_crop_season_zero_rate,31987


In [ ]:
# ============================================================
# MODEL 3 V2 — HISTORICAL FEATURE IMPUTATION
# ============================================================

RISK_FEATURES = [
    "historical_zero_rate_global",
    "historical_crop_zero_rate",
    "historical_state_zero_rate",
    "historical_district_zero_rate",
    "historical_state_crop_zero_rate",
    "historical_district_crop_zero_rate",
    "historical_crop_season_zero_rate",
    "historical_district_crop_season_zero_rate"
]

YIELD_FEATURES = [
    "historical_crop_mean_yield",
    "historical_crop_median_yield",
    "historical_crop_std_yield",
    "historical_crop_min_yield",
    "historical_crop_max_yield",
    "historical_crop_mean_area",
    "historical_crop_median_area"
]

# ------------------------------------------------------------
# GLOBAL HISTORICAL PRIOR
# ------------------------------------------------------------

global_prior = train_expanding[
    "zero_production_flag"
].mean()

print("Global zero-production prior:", global_prior)

# ------------------------------------------------------------
# FILL RISK FEATURES
# ------------------------------------------------------------

for col in RISK_FEATURES:

    train_expanding[col] = (
        train_expanding[col]
        .fillna(global_prior)
    )

# ------------------------------------------------------------
# FILL HISTORICAL NUMERIC FEATURES
# ------------------------------------------------------------

for col in YIELD_FEATURES:

    train_expanding[col] = (
        train_expanding[col]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(
            train_expanding[col].median()
        )
    )

train_expanding["area_vs_historical_crop_mean"] = (
    train_expanding["area_vs_historical_crop_mean"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(1.0)
)

print("\nRemaining missing values:")
display(
    train_expanding[
        RISK_FEATURES +
        YIELD_FEATURES +
        ["area_vs_historical_crop_mean"]
    ].isna().sum()
)

Global zero-production prior: 0.018928482984242014

Remaining missing values:


,0
historical_zero_rate_global,0
historical_crop_zero_rate,0
historical_state_zero_rate,0
historical_district_zero_rate,0
historical_state_crop_zero_rate,0
historical_district_crop_zero_rate,0
historical_crop_season_zero_rate,0
historical_district_crop_season_zero_rate,0
historical_crop_mean_yield,0
historical_crop_median_yield,0


In [ ]:
# ============================================================
# MODEL 3 V2 — VALIDATION / TEST HISTORICAL FEATURES
# ============================================================

# ------------------------------------------------------------
# VALIDATION
# 2018 predictions use history through 2017
# ------------------------------------------------------------

val_features = build_historical_features(
    current_data=val_raw,
    historical_data=train_raw
)

# ------------------------------------------------------------
# TEST
# 2019 predictions use history through 2018
# ------------------------------------------------------------

history_for_test = pd.concat(
    [train_raw, val_raw],
    ignore_index=True
)

test_features = build_historical_features(
    current_data=test_raw,
    historical_data=history_for_test
)

print("=" * 60)
print("VALIDATION / TEST FEATURES")
print("=" * 60)

print("Validation shape:", val_features.shape)
print("Test shape      :", test_features.shape)

VALIDATION / TEST FEATURES
Validation shape: (18281, 26)
Test shape      : (19246, 26)


In [ ]:
# ============================================================
# MODEL 3 V2 — VALIDATION / TEST IMPUTATION
# ============================================================

def impute_historical_features(
    df,
    risk_prior,
    numeric_medians
):

    df = df.copy()

    # Risk-rate features
    for col in RISK_FEATURES:
        df[col] = (
            df[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(risk_prior)
        )

    # Historical numeric features
    for col in YIELD_FEATURES:
        df[col] = (
            df[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(numeric_medians[col])
        )

    df["area_vs_historical_crop_mean"] = (
        df["area_vs_historical_crop_mean"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(1.0)
    )

    return df


numeric_medians = {
    col: train_expanding[col].median()
    for col in YIELD_FEATURES
}

val_features = impute_historical_features(
    val_features,
    global_prior,
    numeric_medians
)

test_features = impute_historical_features(
    test_features,
    global_prior,
    numeric_medians
)

print("=" * 60)
print("FINAL FEATURE SANITY CHECK")
print("=" * 60)

print(
    "Training missing:",
    train_expanding[RISK_FEATURES + YIELD_FEATURES].isna().sum().sum()
)

print(
    "Validation missing:",
    val_features[RISK_FEATURES + YIELD_FEATURES].isna().sum().sum()
)

print(
    "Test missing:",
    test_features[RISK_FEATURES + YIELD_FEATURES].isna().sum().sum()
)

FINAL FEATURE SANITY CHECK
Training missing: 0
Validation missing: 0
Test missing: 0


In [ ]:
# ============================================================
# MODEL 3 V2 — FINAL FEATURE SET
# ============================================================

BASE_FEATURES_M3 = [
    "state",
    "district",
    "crop",
    "season",
    "area",
    "log_area"
]

HISTORICAL_FEATURES_M3 = [
    "historical_zero_rate_global",
    "historical_crop_zero_rate",
    "historical_state_zero_rate",
    "historical_district_zero_rate",
    "historical_state_crop_zero_rate",
    "historical_district_crop_zero_rate",
    "historical_crop_season_zero_rate",
    "historical_district_crop_season_zero_rate",

    "historical_crop_mean_yield",
    "historical_crop_median_yield",
    "historical_crop_std_yield",
    "historical_crop_min_yield",
    "historical_crop_max_yield",

    "historical_crop_mean_area",
    "historical_crop_median_area",

    "area_vs_historical_crop_mean"
]

MODEL_3_V2_FEATURES = (
    BASE_FEATURES_M3 +
    HISTORICAL_FEATURES_M3
)

TARGET_M3 = "zero_production_flag"

print("=" * 60)
print("MODEL 3 V2 — FINAL FEATURES")
print("=" * 60)

print(f"Total features: {len(MODEL_3_V2_FEATURES)}")

for i, feature in enumerate(
    MODEL_3_V2_FEATURES,
    start=1
):
    print(f"{i:2d}. {feature}")

MODEL 3 V2 — FINAL FEATURES
Total features: 22
 1. state
 2. district
 3. crop
 4. season
 5. area
 6. log_area
 7. historical_zero_rate_global
 8. historical_crop_zero_rate
 9. historical_state_zero_rate
10. historical_district_zero_rate
11. historical_state_crop_zero_rate
12. historical_district_crop_zero_rate
13. historical_crop_season_zero_rate
14. historical_district_crop_season_zero_rate
15. historical_crop_mean_yield
16. historical_crop_median_yield
17. historical_crop_std_yield
18. historical_crop_min_yield
19. historical_crop_max_yield
20. historical_crop_mean_area
21. historical_crop_median_area
22. area_vs_historical_crop_mean


In [ ]:
# ============================================================
# MODEL 3 V2 — BUILD TRAINING MATRICES
# ============================================================

X3_train = train_expanding[
    MODEL_3_V2_FEATURES
].copy()

y3_train = train_expanding[
    TARGET_M3
].copy()

X3_val = val_features[
    MODEL_3_V2_FEATURES
].copy()

y3_val = val_features[
    TARGET_M3
].copy()

X3_test = test_features[
    MODEL_3_V2_FEATURES
].copy()

y3_test = test_features[
    TARGET_M3
].copy()

print("=" * 60)
print("MODEL 3 V2 — MATRICES")
print("=" * 60)

print("Train:", X3_train.shape)
print("Val  :", X3_val.shape)
print("Test :", X3_test.shape)

MODEL 3 V2 — MATRICES
Train: (306892, 22)
Val  : (18281, 22)
Test : (19246, 22)


In [ ]:
# ============================================================
# MODEL 3 V2 — CATEGORICAL FEATURES
# ============================================================

CAT_FEATURES_M3 = [
    "state",
    "district",
    "crop",
    "season"
]

CAT_INDICES_M3 = [
    MODEL_3_V2_FEATURES.index(col)
    for col in CAT_FEATURES_M3
]

print("Categorical features:")

for col in CAT_FEATURES_M3:
    print("✓", col)

print("\nCategorical indices:")
print(CAT_INDICES_M3)

Categorical features:
✓ state
✓ district
✓ crop
✓ season

Categorical indices:
[0, 1, 2, 3]


In [ ]:
# ============================================================
# MODEL 3 V2 — CATBOOST TRAINING
# ============================================================

from catboost import CatBoostClassifier

model_3_v2 = CatBoostClassifier(
    iterations=1200,
    learning_rate=0.04,
    depth=8,

    loss_function="Logloss",
    eval_metric="AUC",

    auto_class_weights="Balanced",

    random_seed=42,

    l2_leaf_reg=5,

    verbose=100,
    allow_writing_files=False
)

model_3_v2.fit(
    X3_train,
    y3_train,

    cat_features=CAT_INDICES_M3,

    eval_set=(
        X3_val,
        y3_val
    ),

    use_best_model=True,
    early_stopping_rounds=120
)

print("=" * 60)
print("MODEL 3 V2 — TRAINING COMPLETED")
print("=" * 60)

print(
    "Best iteration:",
    model_3_v2.get_best_iteration()
)

print(
    "Best validation score:",
    model_3_v2.get_best_score()
)

0:	test: 0.9143447	best: 0.9143447 (0)	total: 383ms	remaining: 7m 38s
100:	test: 0.9529770	best: 0.9533102 (96)	total: 36.6s	remaining: 6m 38s
200:	test: 0.9552701	best: 0.9552831 (170)	total: 1m 12s	remaining: 5m 59s
300:	test: 0.9556711	best: 0.9569085 (263)	total: 1m 50s	remaining: 5m 29s
Stopped by overfitting detector  (120 iterations wait)

bestTest = 0.9569084772
bestIteration = 263

Shrink model to first 264 iterations.
MODEL 3 V2 — TRAINING COMPLETED
Best iteration: 263
Best validation score: {'learn': {'Logloss': 0.09474701151754954}, 'validation': {'Logloss': 0.2725912458464477, 'AUC': 0.9569084772181567}}


In [ ]:
# ============================================================
# MODEL 3 V2 — PROBABILITY PREDICTIONS
# ============================================================

val_prob_v2 = (
    model_3_v2
    .predict_proba(X3_val)[:, 1]
)

test_prob_v2 = (
    model_3_v2
    .predict_proba(X3_test)[:, 1]
)

print("=" * 60)
print("MODEL 3 V2 — PROBABILITY OUTPUT")
print("=" * 60)

print("\nValidation:")
print(
    pd.Series(val_prob_v2).describe()
)

print("\nTest:")
print(
    pd.Series(test_prob_v2).describe()
)

MODEL 3 V2 — PROBABILITY OUTPUT

Validation:
count    18281.000000
mean         0.117147
std          0.219575
min          0.000111
25%          0.005893
50%          0.023658
75%          0.097261
max          0.999423
dtype: float64

Test:
count    19246.000000
mean         0.124102
std          0.225511
min          0.000108
25%          0.006253
50%          0.025993
75%          0.109116
max          0.999716
dtype: float64


In [ ]:
# ============================================================
# MODEL 3 V2 — AUC COMPARISON
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

v2_val_roc_auc = roc_auc_score(
    y3_val,
    val_prob_v2
)

v2_test_roc_auc = roc_auc_score(
    y3_test,
    test_prob_v2
)

v2_val_pr_auc = average_precision_score(
    y3_val,
    val_prob_v2
)

v2_test_pr_auc = average_precision_score(
    y3_test,
    test_prob_v2
)

print("=" * 60)
print("MODEL 3 V2 — AUC COMPARISON")
print("=" * 60)

print("\n                    Baseline      V2")

print(
    f"Validation ROC-AUC   0.9446      {v2_val_roc_auc:.4f}"
)

print(
    f"Test ROC-AUC         0.9077      {v2_test_roc_auc:.4f}"
)

print(
    f"Validation PR-AUC    ---         {v2_val_pr_auc:.4f}"
)

print(
    f"Test PR-AUC          0.2141      {v2_test_pr_auc:.4f}"
)

MODEL 3 V2 — AUC COMPARISON

                    Baseline      V2
Validation ROC-AUC   0.9446      0.9569
Test ROC-AUC         0.9077      0.9309
Validation PR-AUC    ---         0.3344
Test PR-AUC          0.2141      0.2688


In [ ]:
# ============================================================
# MODEL 3 V2 — THRESHOLD OPTIMIZATION
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

thresholds_v2 = np.arange(0.05, 0.96, 0.05)

threshold_results_v2 = []

for threshold in thresholds_v2:

    val_pred = (
        val_prob_v2 >= threshold
    ).astype(int)

    precision = precision_score(
        y3_val,
        val_pred,
        zero_division=0
    )

    recall = recall_score(
        y3_val,
        val_pred,
        zero_division=0
    )

    f1 = f1_score(
        y3_val,
        val_pred,
        zero_division=0
    )

    threshold_results_v2.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results_v2 = pd.DataFrame(
    threshold_results_v2
)

display(
    threshold_results_v2.round(4)
)

,threshold,precision,recall,f1
0,0.05,0.0444,0.9828,0.0849
1,0.10,0.0622,0.9622,0.1169
2,0.15,0.0766,0.9485,0.1418
3,0.20,0.0911,0.9313,0.1660
4,0.25,0.1047,0.9175,0.1879
5,0.30,0.1172,0.9072,0.2076
6,0.35,0.1312,0.9038,0.2291
7,0.40,0.1446,0.8866,0.2487
8,0.45,0.1551,0.8591,0.2627
9,0.50,0.1683,0.8282,0.2797


In [ ]:
# ============================================================
# MODEL 3 V2 — FINAL 2019 TEST AT FROZEN THRESHOLD
# ============================================================

BEST_THRESHOLD_V2 = 0.90

test_pred_v2 = (
    test_prob_v2 >= BEST_THRESHOLD_V2
).astype(int)

test_precision_v2 = precision_score(
    y3_test,
    test_pred_v2,
    zero_division=0
)

test_recall_v2 = recall_score(
    y3_test,
    test_pred_v2,
    zero_division=0
)

test_f1_v2 = f1_score(
    y3_test,
    test_pred_v2,
    zero_division=0
)

test_roc_auc_v2 = roc_auc_score(
    y3_test,
    test_prob_v2
)

test_pr_auc_v2 = average_precision_score(
    y3_test,
    test_prob_v2
)

print("=" * 60)
print("MODEL 3 V2 — FINAL TEST METRICS")
print("=" * 60)

print(f"Threshold : {BEST_THRESHOLD_V2:.2f}")
print(f"Precision : {test_precision_v2:.4f}")
print(f"Recall    : {test_recall_v2:.4f}")
print(f"F1        : {test_f1_v2:.4f}")
print(f"ROC-AUC   : {test_roc_auc_v2:.4f}")
print(f"PR-AUC    : {test_pr_auc_v2:.4f}")

MODEL 3 V2 — FINAL TEST METRICS
Threshold : 0.90
Precision : 0.3473
Recall    : 0.5939
F1        : 0.4383
ROC-AUC   : 0.9309
PR-AUC    : 0.2688


In [ ]:
# ============================================================
# MODEL 3 V2 — FINAL CONFUSION MATRIX
# ============================================================

cm_v2 = confusion_matrix(
    y3_test,
    test_pred_v2
)

print("=" * 60)
print("MODEL 3 V2 — FINAL CONFUSION MATRIX")
print("=" * 60)

print("""
                  Predicted
                  Normal   Risk

Actual Normal       TN      FP
Actual Risk         FN      TP
""")

print(cm_v2)

print("\nTN:", cm_v2[0, 0])
print("FP:", cm_v2[0, 1])
print("FN:", cm_v2[1, 0])
print("TP:", cm_v2[1, 1])

MODEL 3 V2 — FINAL CONFUSION MATRIX

                  Predicted
                  Normal   Risk

Actual Normal       TN      FP
Actual Risk         FN      TP

[[18626   327]
 [  119   174]]

TN: 18626
FP: 327
FN: 119
TP: 174


In [ ]:
# ============================================================
# MODEL 3 V3 — FEATURE ENGINEERING SETUP
# ============================================================

print("=" * 60)
print("MODEL 3 V3 — ENHANCED HISTORICAL FEATURES")
print("=" * 60)

print("Base dataset:", apy.shape)
print("Training:", train_raw.shape)
print("Validation:", val_raw.shape)
print("Test:", test_raw.shape)

MODEL 3 V3 — ENHANCED HISTORICAL FEATURES
Base dataset: (344727, 9)
Training: (306892, 9)
Validation: (18281, 9)
Test: (19246, 9)


In [ ]:
# ============================================================
# MODEL 3 V3 — ENHANCED TEMPORAL FEATURE BUILDER
# ============================================================

def build_v3_features(current_data, historical_data):
    """
    Build leakage-safe historical risk features.

    All historical features are calculated exclusively from
    historical_data, which must contain observations strictly
    earlier than current_data.

    Features:
        - Long-term failure rates
        - Recent 3/5/10-year failure rates
        - Historical sample counts
        - District/crop yield stability
        - Area volatility
        - Current area abnormality
    """

    current = current_data.copy()
    history = historical_data.copy()

    current_years = sorted(
        current["crop_year"].dropna().unique()
    )

    feature_rows = []

    for year in current_years:

        current_year_data = current[
            current["crop_year"] == year
        ].copy()

        # Strictly historical information
        hist = history[
            history["crop_year"] < year
        ].copy()

        # ----------------------------------------------------
        # BASIC CURRENT FEATURES
        # ----------------------------------------------------

        current_year_data["log_area"] = np.log1p(
            current_year_data["area"].clip(lower=0)
        )

        # ----------------------------------------------------
        # NO HISTORY
        # ----------------------------------------------------

        if hist.empty:

            # Risk
            current_year_data[
                "historical_zero_rate_global"
            ] = np.nan

            current_year_data[
                "recent_3yr_zero_rate"
            ] = np.nan

            current_year_data[
                "recent_5yr_zero_rate"
            ] = np.nan

            current_year_data[
                "recent_10yr_zero_rate"
            ] = np.nan

            # Counts
            count_features = [
                "crop_history_count",
                "state_history_count",
                "district_history_count",
                "state_crop_history_count",
                "district_crop_history_count",
                "district_crop_season_history_count"
            ]

            for col in count_features:
                current_year_data[col] = np.nan

            # Yield / area statistics
            numeric_features = [
                "district_crop_mean_yield",
                "district_crop_median_yield",
                "district_crop_std_yield",
                "district_crop_min_yield",
                "district_crop_max_yield",
                "district_crop_mean_area",
                "district_crop_std_area",
                "district_crop_cv_area",
                "recent_3yr_area_mean",
                "recent_5yr_area_mean",
                "area_vs_recent_3yr_mean",
                "area_vs_recent_5yr_mean"
            ]

            for col in numeric_features:
                current_year_data[col] = np.nan

            feature_rows.append(current_year_data)
            continue

        # ----------------------------------------------------
        # GLOBAL HISTORICAL RISK
        # ----------------------------------------------------

        global_rate = (
            hist["zero_production_flag"].mean()
        )

        current_year_data[
            "historical_zero_rate_global"
        ] = global_rate

        # ----------------------------------------------------
        # RECENT WINDOWS
        # ----------------------------------------------------

        def recent_rate(window):

            recent = hist[
                hist["crop_year"] >=
                year - window
            ]

            if recent.empty:
                return np.nan

            return recent[
                "zero_production_flag"
            ].mean()

        current_year_data[
            "recent_3yr_zero_rate"
        ] = recent_rate(3)

        current_year_data[
            "recent_5yr_zero_rate"
        ] = recent_rate(5)

        current_year_data[
            "recent_10yr_zero_rate"
        ] = recent_rate(10)

        # ----------------------------------------------------
        # LONG-TERM RISK BY CROP
        # ----------------------------------------------------

        crop_rate = (
            hist.groupby("crop")
            ["zero_production_flag"]
            .mean()
        )

        current_year_data[
            "historical_crop_zero_rate"
        ] = current_year_data["crop"].map(
            crop_rate
        )

        # ----------------------------------------------------
        # LONG-TERM RISK BY STATE
        # ----------------------------------------------------

        state_rate = (
            hist.groupby("state")
            ["zero_production_flag"]
            .mean()
        )

        current_year_data[
            "historical_state_zero_rate"
        ] = current_year_data["state"].map(
            state_rate
        )

        # ----------------------------------------------------
        # LONG-TERM RISK BY DISTRICT
        # ----------------------------------------------------

        district_rate = (
            hist.groupby("district")
            ["zero_production_flag"]
            .mean()
        )

        current_year_data[
            "historical_district_zero_rate"
        ] = current_year_data["district"].map(
            district_rate
        )

        # ----------------------------------------------------
        # STATE + CROP
        # ----------------------------------------------------

        state_crop_rate = (
            hist.groupby(
                ["state", "crop"]
            )["zero_production_flag"]
            .mean()
        )

        current_year_data[
            "historical_state_crop_zero_rate"
        ] = [
            state_crop_rate.get(
                (s, c),
                np.nan
            )
            for s, c in zip(
                current_year_data["state"],
                current_year_data["crop"]
            )
        ]

        # ----------------------------------------------------
        # DISTRICT + CROP
        # ----------------------------------------------------

        district_crop_rate = (
            hist.groupby(
                ["district", "crop"]
            )["zero_production_flag"]
            .mean()
        )

        current_year_data[
            "historical_district_crop_zero_rate"
        ] = [
            district_crop_rate.get(
                (d, c),
                np.nan
            )
            for d, c in zip(
                current_year_data["district"],
                current_year_data["crop"]
            )
        ]

        # ----------------------------------------------------
        # CROP + SEASON
        # ----------------------------------------------------

        crop_season_rate = (
            hist.groupby(
                ["crop", "season"]
            )["zero_production_flag"]
            .mean()
        )

        current_year_data[
            "historical_crop_season_zero_rate"
        ] = [
            crop_season_rate.get(
                (c, s),
                np.nan
            )
            for c, s in zip(
                current_year_data["crop"],
                current_year_data["season"]
            )
        ]

        # ----------------------------------------------------
        # DISTRICT + CROP + SEASON
        # ----------------------------------------------------

        dcs_rate = (
            hist.groupby(
                ["district", "crop", "season"]
            )["zero_production_flag"]
            .mean()
        )

        current_year_data[
            "historical_district_crop_season_zero_rate"
        ] = [
            dcs_rate.get(
                (d, c, s),
                np.nan
            )
            for d, c, s in zip(
                current_year_data["district"],
                current_year_data["crop"],
                current_year_data["season"]
            )
        ]

        # ----------------------------------------------------
        # HISTORICAL SAMPLE COUNTS
        # ----------------------------------------------------

        current_year_data[
            "crop_history_count"
        ] = current_year_data["crop"].map(
            hist["crop"].value_counts()
        )

        current_year_data[
            "state_history_count"
        ] = current_year_data["state"].map(
            hist["state"].value_counts()
        )

        current_year_data[
            "district_history_count"
        ] = current_year_data["district"].map(
            hist["district"].value_counts()
        )

        state_crop_count = (
            hist.groupby(
                ["state", "crop"]
            ).size()
        )

        current_year_data[
            "state_crop_history_count"
        ] = [
            state_crop_count.get(
                (s, c),
                np.nan
            )
            for s, c in zip(
                current_year_data["state"],
                current_year_data["crop"]
            )
        ]

        district_crop_count = (
            hist.groupby(
                ["district", "crop"]
            ).size()
        )

        current_year_data[
            "district_crop_history_count"
        ] = [
            district_crop_count.get(
                (d, c),
                np.nan
            )
            for d, c in zip(
                current_year_data["district"],
                current_year_data["crop"]
            )
        ]

        dcs_count = (
            hist.groupby(
                ["district", "crop", "season"]
            ).size()
        )

        current_year_data[
            "district_crop_season_history_count"
        ] = [
            dcs_count.get(
                (d, c, s),
                np.nan
            )
            for d, c, s in zip(
                current_year_data["district"],
                current_year_data["crop"],
                current_year_data["season"]
            )
        ]

        # ----------------------------------------------------
        # DISTRICT + CROP YIELD STABILITY
        # ----------------------------------------------------

        dc_yield = (
            hist.groupby(
                ["district", "crop"]
            )["yield"]
            .agg(
                mean="mean",
                median="median",
                std="std",
                min="min",
                max="max"
            )
        )

        current_year_data[
            "district_crop_mean_yield"
        ] = [
            dc_yield["mean"].get(
                (d, c),
                np.nan
            )
            for d, c in zip(
                current_year_data["district"],
                current_year_data["crop"]
            )
        ]

        current_year_data[
            "district_crop_median_yield"
        ] = [
            dc_yield["median"].get(
                (d, c),
                np.nan
            )
            for d, c in zip(
                current_year_data["district"],
                current_year_data["crop"]
            )
        ]

        current_year_data[
            "district_crop_std_yield"
        ] = [
            dc_yield["std"].get(
                (d, c),
                np.nan
            )
            for d, c in zip(
                current_year_data["district"],
                current_year_data["crop"]
            )
        ]

        current_year_data[
            "district_crop_min_yield"
        ] = [
            dc_yield["min"].get(
                (d, c),
                np.nan
            )
            for d, c in zip(
                current_year_data["district"],
                current_year_data["crop"]
            )
        ]

        current_year_data[
            "district_crop_max_yield"
        ] = [
            dc_yield["max"].get(
                (d, c),
                np.nan
            )
            for d, c in zip(
                current_year_data["district"],
                current_year_data["crop"]
            )
        ]

        # ----------------------------------------------------
        # DISTRICT + CROP AREA STATISTICS
        # ----------------------------------------------------

        dc_area = (
            hist.groupby(
                ["district", "crop"]
            )["area"]
            .agg(
                mean="mean",
                std="std"
            )
        )

        current_year_data[
            "district_crop_mean_area"
        ] = [
            dc_area["mean"].get(
                (d, c),
                np.nan
            )
            for d, c in zip(
                current_year_data["district"],
                current_year_data["crop"]
            )
        ]

        current_year_data[
            "district_crop_std_area"
        ] = [
            dc_area["std"].get(
                (d, c),
                np.nan
            )
            for d, c in zip(
                current_year_data["district"],
                current_year_data["crop"]
            )
        ]

        current_year_data[
            "district_crop_cv_area"
        ] = (
            current_year_data[
                "district_crop_std_area"
            ]
            /
            (
                current_year_data[
                    "district_crop_mean_area"
                ] + 1e-6
            )
        )

        # ----------------------------------------------------
        # RECENT AREA BEHAVIOUR
        # ----------------------------------------------------

        recent_3 = hist[
            hist["crop_year"] >= year - 3
        ]

        recent_5 = hist[
            hist["crop_year"] >= year - 5
        ]

        r3_area = (
            recent_3
            .groupby(
                ["district", "crop"]
            )["area"]
            .mean()
        )

        r5_area = (
            recent_5
            .groupby(
                ["district", "crop"]
            )["area"]
            .mean()
        )

        current_year_data[
            "recent_3yr_area_mean"
        ] = [
            r3_area.get(
                (d, c),
                np.nan
            )
            for d, c in zip(
                current_year_data["district"],
                current_year_data["crop"]
            )
        ]

        current_year_data[
            "recent_5yr_area_mean"
        ] = [
            r5_area.get(
                (d, c),
                np.nan
            )
            for d, c in zip(
                current_year_data["district"],
                current_year_data["crop"]
            )
        ]

        current_year_data[
            "area_vs_recent_3yr_mean"
        ] = (
            current_year_data["area"]
            /
            (
                current_year_data[
                    "recent_3yr_area_mean"
                ] + 1e-6
            )
        )

        current_year_data[
            "area_vs_recent_5yr_mean"
        ] = (
            current_year_data["area"]
            /
            (
                current_year_data[
                    "recent_5yr_area_mean"
                ] + 1e-6
            )
        )

        feature_rows.append(
            current_year_data
        )

    result = pd.concat(
        feature_rows,
        ignore_index=True
    )

    return result

In [ ]:
# ============================================================
# BUILD MODEL 3 V3 TRAINING FEATURES
# ============================================================

train_v3 = build_v3_features(
    current_data=train_raw,
    historical_data=train_raw
)

print("=" * 60)
print("MODEL 3 V3 — TRAINING FEATURES CREATED")
print("=" * 60)

print("Shape:", train_v3.shape)

print("\nNumber of columns:", len(train_v3.columns))

print("\nNew V3 features:")

v3_new_features = [
    col for col in train_v3.columns
    if col not in apy.columns
]

for feature in v3_new_features:
    print("✓", feature)

MODEL 3 V3 — TRAINING FEATURES CREATED
Shape: (306892, 39)

Number of columns: 39

New V3 features:
✓ log_area
✓ historical_zero_rate_global
✓ recent_3yr_zero_rate
✓ recent_5yr_zero_rate
✓ recent_10yr_zero_rate
✓ crop_history_count
✓ state_history_count
✓ district_history_count
✓ state_crop_history_count
✓ district_crop_history_count
✓ district_crop_season_history_count
✓ district_crop_mean_yield
✓ district_crop_median_yield
✓ district_crop_std_yield
✓ district_crop_min_yield
✓ district_crop_max_yield
✓ district_crop_mean_area
✓ district_crop_std_area
✓ district_crop_cv_area
✓ recent_3yr_area_mean
✓ recent_5yr_area_mean
✓ area_vs_recent_3yr_mean
✓ area_vs_recent_5yr_mean
✓ historical_crop_zero_rate
✓ historical_state_zero_rate
✓ historical_district_zero_rate
✓ historical_state_crop_zero_rate
✓ historical_district_crop_zero_rate
✓ historical_crop_season_zero_rate
✓ historical_district_crop_season_zero_rate


In [ ]:
# ============================================================
# MODEL 3 V3 — VALIDATION / TEST FEATURES
# ============================================================

val_v3 = build_v3_features(
    current_data=val_raw,
    historical_data=train_raw
)

history_for_test_v3 = pd.concat(
    [train_raw, val_raw],
    ignore_index=True
)

test_v3 = build_v3_features(
    current_data=test_raw,
    historical_data=history_for_test_v3
)

print("=" * 60)
print("MODEL 3 V3 — VALIDATION / TEST FEATURES")
print("=" * 60)

print("Training   :", train_v3.shape)
print("Validation :", val_v3.shape)
print("Test       :", test_v3.shape)

MODEL 3 V3 — VALIDATION / TEST FEATURES
Training   : (306892, 39)
Validation : (18281, 39)
Test       : (19246, 39)


In [ ]:
# ============================================================
# MODEL 3 V3 — MISSING VALUE CHECK
# ============================================================

v3_feature_columns = [
    col for col in train_v3.columns
    if col not in apy.columns
]

print("=" * 60)
print("MODEL 3 V3 — MISSINGNESS")
print("=" * 60)

print("\nTraining missing:")
display(
    train_v3[v3_feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\nValidation missing:")
display(
    val_v3[v3_feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\nTest missing:")
display(
    test_v3[v3_feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

MODEL 3 V3 — MISSINGNESS

Training missing:


,0
district_crop_std_yield,40942
district_crop_std_area,40942
district_crop_cv_area,40942
historical_district_crop_season_zero_rate,31987
district_crop_season_history_count,31987
recent_3yr_area_mean,27718
area_vs_recent_3yr_mean,27718
recent_5yr_area_mean,25395
area_vs_recent_5yr_mean,25395
district_crop_mean_area,23424



Validation missing:


,0
recent_3yr_area_mean,743
area_vs_recent_3yr_mean,743
district_crop_cv_area,665
district_crop_std_area,665
district_crop_std_yield,665
recent_5yr_area_mean,568
area_vs_recent_5yr_mean,568
historical_district_crop_season_zero_rate,435
district_crop_season_history_count,435
district_crop_mean_area,319



Test missing:


,0
recent_3yr_area_mean,1255
area_vs_recent_3yr_mean,1255
area_vs_recent_5yr_mean,1141
recent_5yr_area_mean,1141
district_crop_season_history_count,785
historical_district_crop_season_zero_rate,785
district_crop_std_area,750
district_crop_cv_area,750
district_crop_std_yield,750
district_crop_mean_area,421


In [ ]:
# ============================================================
# MODEL 3 V3 — LEAKAGE-SAFE IMPUTATION
# ============================================================

# ------------------------------------------------------------
# FEATURE GROUPS
# ------------------------------------------------------------

RISK_FEATURES_V3 = [
    "historical_zero_rate_global",
    "historical_crop_zero_rate",
    "historical_state_zero_rate",
    "historical_district_zero_rate",
    "historical_state_crop_zero_rate",
    "historical_district_crop_zero_rate",
    "historical_crop_season_zero_rate",
    "historical_district_crop_season_zero_rate",
    "recent_3yr_zero_rate",
    "recent_5yr_zero_rate",
    "recent_10yr_zero_rate"
]

COUNT_FEATURES_V3 = [
    "crop_history_count",
    "state_history_count",
    "district_history_count",
    "state_crop_history_count",
    "district_crop_history_count",
    "district_crop_season_history_count"
]

NUMERIC_HISTORY_FEATURES_V3 = [
    "district_crop_mean_yield",
    "district_crop_median_yield",
    "district_crop_std_yield",
    "district_crop_min_yield",
    "district_crop_max_yield",
    "district_crop_mean_area",
    "district_crop_std_area",
    "district_crop_cv_area",
    "recent_3yr_area_mean",
    "recent_5yr_area_mean",
    "area_vs_recent_3yr_mean",
    "area_vs_recent_5yr_mean"
]

# ------------------------------------------------------------
# GLOBAL PRIOR FROM TRAINING ONLY
# ------------------------------------------------------------

GLOBAL_ZERO_PRIOR_V3 = (
    train_raw["zero_production_flag"].mean()
)

print("=" * 60)
print("MODEL 3 V3 — IMPUTATION PARAMETERS")
print("=" * 60)

print(
    f"Global zero-production prior: "
    f"{GLOBAL_ZERO_PRIOR_V3:.6f}"
)

# ------------------------------------------------------------
# TRAINING MEDIANS
# ------------------------------------------------------------

TRAIN_MEDIANS_V3 = {}

for col in NUMERIC_HISTORY_FEATURES_V3:

    TRAIN_MEDIANS_V3[col] = (
        train_v3[col]
        .replace([np.inf, -np.inf], np.nan)
        .median()
    )

# ------------------------------------------------------------
# IMPUTATION FUNCTION
# ------------------------------------------------------------

def impute_v3_features(df):

    df = df.copy()

    # Risk rates
    for col in RISK_FEATURES_V3:

        df[col] = (
            df[col]
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
            .fillna(
                GLOBAL_ZERO_PRIOR_V3
            )
        )

    # Historical counts
    # NaN means no historical observations
    for col in COUNT_FEATURES_V3:

        df[col] = (
            df[col]
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
            .fillna(0)
        )

    # Historical numeric statistics
    for col in NUMERIC_HISTORY_FEATURES_V3:

        df[col] = (
            df[col]
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
            .fillna(
                TRAIN_MEDIANS_V3[col]
            )
        )

    return df

MODEL 3 V3 — IMPUTATION PARAMETERS
Global zero-production prior: 0.018928


In [ ]:
# ============================================================
# APPLY V3 IMPUTATION
# ============================================================

train_v3 = impute_v3_features(train_v3)
val_v3 = impute_v3_features(val_v3)
test_v3 = impute_v3_features(test_v3)

print("=" * 60)
print("MODEL 3 V3 — FINAL FEATURE SANITY CHECK")
print("=" * 60)

all_v3_features = (
    RISK_FEATURES_V3 +
    COUNT_FEATURES_V3 +
    NUMERIC_HISTORY_FEATURES_V3
)

print(
    "Training missing:",
    train_v3[all_v3_features]
    .isna()
    .sum()
    .sum()
)

print(
    "Validation missing:",
    val_v3[all_v3_features]
    .isna()
    .sum()
    .sum()
)

print(
    "Test missing:",
    test_v3[all_v3_features]
    .isna()
    .sum()
    .sum()
)

MODEL 3 V3 — FINAL FEATURE SANITY CHECK
Training missing: 0
Validation missing: 0
Test missing: 0


In [ ]:
# ============================================================
# MODEL 3 V3 — FINAL MATRICES
# ============================================================

BASE_FEATURES_V3 = [
    "state",
    "district",
    "crop",
    "season",
    "area",
    "log_area"
]

MODEL_3_V3_FEATURES = (
    BASE_FEATURES_V3 +
    RISK_FEATURES_V3 +
    COUNT_FEATURES_V3 +
    NUMERIC_HISTORY_FEATURES_V3
)

TARGET_V3 = "zero_production_flag"

X3v3_train = train_v3[
    MODEL_3_V3_FEATURES
].copy()

y3v3_train = train_v3[
    TARGET_V3
].copy()

X3v3_val = val_v3[
    MODEL_3_V3_FEATURES
].copy()

y3v3_val = val_v3[
    TARGET_V3
].copy()

X3v3_test = test_v3[
    MODEL_3_V3_FEATURES
].copy()

y3v3_test = test_v3[
    TARGET_V3
].copy()

print("=" * 60)
print("MODEL 3 V3 — FINAL MATRICES")
print("=" * 60)

print("Feature count:", len(MODEL_3_V3_FEATURES))

print("Train:", X3v3_train.shape)
print("Val  :", X3v3_val.shape)
print("Test :", X3v3_test.shape)

MODEL 3 V3 — FINAL MATRICES
Feature count: 35
Train: (306892, 35)
Val  : (18281, 35)
Test : (19246, 35)


In [ ]:
# ============================================================
# MODEL 3 V3 — CATEGORICAL FEATURES
# ============================================================

from catboost import CatBoostClassifier

CAT_FEATURES_V3 = [
    "state",
    "district",
    "crop",
    "season"
]

CAT_INDICES_V3 = [
    MODEL_3_V3_FEATURES.index(col)
    for col in CAT_FEATURES_V3
]

print("=" * 60)
print("MODEL 3 V3 — CATEGORICAL FEATURES")
print("=" * 60)

for col, idx in zip(
    CAT_FEATURES_V3,
    CAT_INDICES_V3
):
    print(f"{col:12s} → index {idx}")

MODEL 3 V3 — CATEGORICAL FEATURES
state        → index 0
district     → index 1
crop         → index 2
season       → index 3


In [ ]:
# ============================================================
# MODEL 3 V3 — TRAINING
# ============================================================

model_3_v3 = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.035,
    depth=8,

    loss_function="Logloss",
    eval_metric="AUC",

    auto_class_weights="Balanced",

    l2_leaf_reg=7,

    random_seed=42,

    random_strength=1.0,

    verbose=100,
    allow_writing_files=False
)

model_3_v3.fit(
    X3v3_train,
    y3v3_train,

    cat_features=CAT_INDICES_V3,

    eval_set=(
        X3v3_val,
        y3v3_val
    ),

    use_best_model=True,

    early_stopping_rounds=150
)

print("=" * 60)
print("MODEL 3 V3 — TRAINING COMPLETED")
print("=" * 60)

print(
    "Best iteration:",
    model_3_v3.get_best_iteration()
)

print(
    "Best validation score:",
    model_3_v3.get_best_score()
)

0:	test: 0.9336671	best: 0.9336671 (0)	total: 1.08s	remaining: 26m 56s
100:	test: 0.9547324	best: 0.9551750 (98)	total: 1m 19s	remaining: 18m 26s
200:	test: 0.9601611	best: 0.9601627 (199)	total: 2m 13s	remaining: 14m 24s
300:	test: 0.9606652	best: 0.9609203 (286)	total: 3m 9s	remaining: 12m 32s
400:	test: 0.9609755	best: 0.9611881 (389)	total: 4m 1s	remaining: 11m 1s
500:	test: 0.9607015	best: 0.9614920 (465)	total: 4m 54s	remaining: 9m 47s
600:	test: 0.9611982	best: 0.9615187 (539)	total: 5m 49s	remaining: 8m 42s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.9615187131
bestIteration = 539

Shrink model to first 540 iterations.
MODEL 3 V3 — TRAINING COMPLETED
Best iteration: 539
Best validation score: {'learn': {'Logloss': 0.06458960944786242}, 'validation': {'Logloss': 0.26861513055241876, 'AUC': 0.9615187131453328}}


In [ ]:
# ============================================================
# MODEL 3 V3 — PREDICTIONS
# ============================================================

val_prob_v3 = (
    model_3_v3
    .predict_proba(X3v3_val)[:, 1]
)

test_prob_v3 = (
    model_3_v3
    .predict_proba(X3v3_test)[:, 1]
)

print("=" * 60)
print("MODEL 3 V3 — PROBABILITY STATISTICS")
print("=" * 60)

print("\nValidation:")
print(
    pd.Series(val_prob_v3).describe()
)

print("\nTest:")
print(
    pd.Series(test_prob_v3).describe()
)

MODEL 3 V3 — PROBABILITY STATISTICS

Validation:
count    18281.000000
mean         0.073841
std          0.179319
min          0.000015
25%          0.001388
50%          0.007881
75%          0.043110
max          0.998384
dtype: float64

Test:
count    19246.000000
mean         0.075281
std          0.179578
min          0.000015
25%          0.001476
50%          0.008304
75%          0.046243
max          0.998197
dtype: float64


In [ ]:
# ============================================================
# MODEL 3 V3 — AUC / PR-AUC
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

v3_val_roc_auc = roc_auc_score(
    y3v3_val,
    val_prob_v3
)

v3_test_roc_auc = roc_auc_score(
    y3v3_test,
    test_prob_v3
)

v3_val_pr_auc = average_precision_score(
    y3v3_val,
    val_prob_v3
)

v3_test_pr_auc = average_precision_score(
    y3v3_test,
    test_prob_v3
)

print("=" * 60)
print("MODEL 3 V3 — AUC COMPARISON")
print("=" * 60)

print(
    f"Validation ROC-AUC : {v3_val_roc_auc:.4f}"
)

print(
    f"Test ROC-AUC       : {v3_test_roc_auc:.4f}"
)

print(
    f"Validation PR-AUC  : {v3_val_pr_auc:.4f}"
)

print(
    f"Test PR-AUC        : {v3_test_pr_auc:.4f}"
)

print("\n" + "=" * 60)
print("COMPARISON AGAINST MODEL 3 V2")
print("=" * 60)

print(
    f"V2 Test ROC-AUC : 0.9309"
)

print(
    f"V3 Test ROC-AUC : {v3_test_roc_auc:.4f}"
)

print(
    f"V2 Test PR-AUC  : 0.2688"
)

print(
    f"V3 Test PR-AUC  : {v3_test_pr_auc:.4f}"
)

MODEL 3 V3 — AUC COMPARISON
Validation ROC-AUC : 0.9615
Test ROC-AUC       : 0.9438
Validation PR-AUC  : 0.4968
Test PR-AUC        : 0.4293

COMPARISON AGAINST MODEL 3 V2
V2 Test ROC-AUC : 0.9309
V3 Test ROC-AUC : 0.9438
V2 Test PR-AUC  : 0.2688
V3 Test PR-AUC  : 0.4293


In [ ]:
# ============================================================
# MODEL 3 V3 — THRESHOLD OPTIMIZATION
# ============================================================

thresholds_v3 = np.arange(
    0.01,
    1.00,
    0.01
)

threshold_results_v3 = []

for threshold in thresholds_v3:

    val_pred = (
        val_prob_v3 >= threshold
    ).astype(int)

    precision = precision_score(
        y3v3_val,
        val_pred,
        zero_division=0
    )

    recall = recall_score(
        y3v3_val,
        val_pred,
        zero_division=0
    )

    f1 = f1_score(
        y3v3_val,
        val_pred,
        zero_division=0
    )

    threshold_results_v3.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results_v3 = pd.DataFrame(
    threshold_results_v3
)

display(
    threshold_results_v3
    .sort_values("f1", ascending=False)
    .head(20)
    .round(4)
)

,threshold,precision,recall,f1
89,0.90,0.5151,0.5292,0.5220
91,0.92,0.5370,0.4983,0.5169
92,0.93,0.5605,0.4777,0.5158
85,0.86,0.4650,0.5704,0.5123
90,0.91,0.5194,0.5052,0.5122
93,0.94,0.5808,0.4570,0.5115
88,0.89,0.4890,0.5361,0.5115
84,0.85,0.4528,0.5773,0.5076
86,0.87,0.4667,0.5533,0.5063
83,0.84,0.4419,0.5876,0.5044


In [ ]:
# ============================================================
# MODEL 3 V3 — BEST F1 THRESHOLD
# ============================================================

best_v3_row = (
    threshold_results_v3
    .sort_values("f1", ascending=False)
    .iloc[0]
)

BEST_THRESHOLD_V3 = float(
    best_v3_row["threshold"]
)

print("=" * 60)
print("MODEL 3 V3 — BEST VALIDATION THRESHOLD")
print("=" * 60)

print(
    f"Threshold : {BEST_THRESHOLD_V3:.2f}"
)

print(
    f"Precision : {best_v3_row['precision']:.4f}"
)

print(
    f"Recall    : {best_v3_row['recall']:.4f}"
)

print(
    f"F1        : {best_v3_row['f1']:.4f}"
)

MODEL 3 V3 — BEST VALIDATION THRESHOLD
Threshold : 0.90
Precision : 0.5151
Recall    : 0.5292
F1        : 0.5220


In [ ]:
# ============================================================
# MODEL 3 V3 — FINAL 2019 TEST
# ============================================================

test_pred_v3 = (
    test_prob_v3 >= BEST_THRESHOLD_V3
).astype(int)

test_precision_v3 = precision_score(
    y3v3_test,
    test_pred_v3,
    zero_division=0
)

test_recall_v3 = recall_score(
    y3v3_test,
    test_pred_v3,
    zero_division=0
)

test_f1_v3 = f1_score(
    y3v3_test,
    test_pred_v3,
    zero_division=0
)

print("=" * 60)
print("MODEL 3 V3 — FINAL TEST METRICS")
print("=" * 60)

print(f"Threshold : {BEST_THRESHOLD_V3:.2f}")
print(f"Precision : {test_precision_v3:.4f}")
print(f"Recall    : {test_recall_v3:.4f}")
print(f"F1        : {test_f1_v3:.4f}")
print(f"ROC-AUC   : {v3_test_roc_auc:.4f}")
print(f"PR-AUC    : {v3_test_pr_auc:.4f}")

MODEL 3 V3 — FINAL TEST METRICS
Threshold : 0.90
Precision : 0.4966
Recall    : 0.5017
F1        : 0.4992
ROC-AUC   : 0.9438
PR-AUC    : 0.4293


In [ ]:
# ============================================================
# MODEL 3 V3 — CONFUSION MATRIX
# ============================================================

cm_v3 = confusion_matrix(
    y3v3_test,
    test_pred_v3
)

print("=" * 60)
print("MODEL 3 V3 — CONFUSION MATRIX")
print("=" * 60)

print(cm_v3)

print("\nTN:", cm_v3[0, 0])
print("FP:", cm_v3[0, 1])
print("FN:", cm_v3[1, 0])
print("TP:", cm_v3[1, 1])

MODEL 3 V3 — CONFUSION MATRIX
[[18804   149]
 [  146   147]]

TN: 18804
FP: 149
FN: 146
TP: 147


In [ ]:
# ============================================================
# MODEL 3 V3 — FEATURE IMPORTANCE
# ============================================================

feature_importance_v3 = pd.DataFrame({
    "feature": MODEL_3_V3_FEATURES,
    "importance": model_3_v3.get_feature_importance()
})

feature_importance_v3 = (
    feature_importance_v3
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 60)
print("MODEL 3 V3 — FEATURE IMPORTANCE")
print("=" * 60)

display(
    feature_importance_v3
)

MODEL 3 V3 — FEATURE IMPORTANCE


,feature,importance
0,state,8.551732
1,historical_state_zero_rate,7.982497
2,area,5.687235
3,district_crop_history_count,5.458486
4,log_area,5.381838
5,state_history_count,4.917306
6,historical_state_crop_zero_rate,3.854464
7,district_crop_min_yield,3.690721
8,recent_5yr_zero_rate,3.339376
9,district,2.893257


In [ ]:
# ============================================================
# MODEL 3 V3 — FEATURE IMPORTANCE SUMMARY
# ============================================================

def feature_group(feature):

    if feature in [
        "state",
        "district",
        "crop",
        "season",
        "area",
        "log_area"
    ]:
        return "Current/Input"

    if "recent_" in feature:
        return "Recent History"

    if "history_count" in feature:
        return "Historical Reliability"

    if "yield" in feature:
        return "Yield Stability"

    if "area" in feature:
        return "Area Behaviour"

    if "zero_rate" in feature:
        return "Historical Risk"

    return "Other"


feature_importance_v3["group"] = (
    feature_importance_v3["feature"]
    .apply(feature_group)
)

group_importance = (
    feature_importance_v3
    .groupby("group")["importance"]
    .sum()
    .sort_values(ascending=False)
)

print("=" * 60)
print("MODEL 3 V3 — IMPORTANCE BY FEATURE GROUP")
print("=" * 60)

display(
    group_importance.to_frame("total_importance")
)

MODEL 3 V3 — IMPORTANCE BY FEATURE GROUP


,total_importance
group,
Current/Input,27.789895
Historical Risk,23.385887
Historical Reliability,19.492034
Recent History,13.943251
Yield Stability,11.230640
Area Behaviour,4.158292


In [ ]:
# ============================================================
# MODEL 3 V3 — PRECISION / RECALL CURVE
# ============================================================

from sklearn.metrics import (
    precision_recall_curve,
    auc
)

precision_curve, recall_curve, pr_thresholds = (
    precision_recall_curve(
        y3v3_test,
        test_prob_v3
    )
)

pr_curve_auc = auc(
    recall_curve,
    precision_curve
)

print("=" * 60)
print("MODEL 3 V3 — PR CURVE DIAGNOSTICS")
print("=" * 60)

print(
    f"PR-AUC from curve : {pr_curve_auc:.4f}"
)

print(
    f"Positive prevalence: "
    f"{y3v3_test.mean():.4f}"
)

MODEL 3 V3 — PR CURVE DIAGNOSTICS
PR-AUC from curve : 0.4269
Positive prevalence: 0.0152


In [ ]:
# ============================================================
# TOP-RANKED RISK CASES
# ============================================================

diagnostic_v3 = pd.DataFrame({
    "actual": y3v3_test.reset_index(drop=True),
    "probability": test_prob_v3
})

diagnostic_v3 = (
    diagnostic_v3
    .sort_values(
        "probability",
        ascending=False
    )
    .reset_index(drop=True)
)

diagnostic_v3["rank"] = (
    np.arange(len(diagnostic_v3)) + 1
)

display(
    diagnostic_v3.head(50)
)

,actual,probability,rank
0,0,0.998197,1
1,0,0.997657,2
2,0,0.997417,3
3,0,0.996933,4
4,1,0.996855,5
5,1,0.996814,6
6,1,0.996795,7
7,1,0.996776,8
8,1,0.996390,9
9,0,0.996095,10


In [ ]:
# ============================================================
# PRECISION@K
# ============================================================

print("=" * 60)
print("MODEL 3 V3 — PRECISION@K")
print("=" * 60)

for k in [
    50,
    100,
    200,
    300,
    500,
    1000
]:

    top_k = diagnostic_v3.head(k)

    precision_at_k = (
        top_k["actual"].sum() / k
    )

    recall_at_k = (
        top_k["actual"].sum()
        /
        y3v3_test.sum()
    )

    print(
        f"Top-{k:4d} | "
        f"Precision: {precision_at_k:.4f} | "
        f"Recall: {recall_at_k:.4f}"
    )

MODEL 3 V3 — PRECISION@K
Top-  50 | Precision: 0.7600 | Recall: 0.1297
Top- 100 | Precision: 0.6400 | Recall: 0.2184
Top- 200 | Precision: 0.5900 | Recall: 0.4027
Top- 300 | Precision: 0.4900 | Recall: 0.5017
Top- 500 | Precision: 0.3620 | Recall: 0.6177
Top-1000 | Precision: 0.2130 | Recall: 0.7270


In [ ]:

# ============================================================
# MODEL 3 V3 — CALIBRATION DIAGNOSTICS
# ============================================================

from sklearn.metrics import (
    brier_score_loss
)
from sklearn.calibration import (
    calibration_curve
)

brier_v3 = brier_score_loss(
    y3v3_test,
    test_prob_v3
)

prob_true, prob_pred = calibration_curve(
    y3v3_test,
    test_prob_v3,
    n_bins=10,
    strategy="quantile"
)

print("=" * 60)
print("MODEL 3 V3 — CALIBRATION DIAGNOSTICS")
print("=" * 60)

print(
    f"Brier score: {brier_v3:.6f}"
)

print("\nCalibration bins:")

calibration_df = pd.DataFrame({
    "mean_predicted_probability": prob_pred,
    "actual_positive_rate": prob_true
})

display(
    calibration_df.round(4)
)

MODEL 3 V3 — CALIBRATION DIAGNOSTICS
Brier score: 0.032017

Calibration bins:


,mean_predicted_probability,actual_positive_rate
0,0.0002,0.0000
1,0.0007,0.0000
2,0.0015,0.0005
3,0.0031,0.0010
4,0.0062,0.0000
5,0.0116,0.0026
6,0.0225,0.0047
7,0.0477,0.0047
8,0.1193,0.0146
9,0.5399,0.1242


In [ ]:
# ============================================================
# MODEL 3 V3 — ISOTONIC PROBABILITY CALIBRATION
# ============================================================

from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss

# ------------------------------------------------------------
# Fit ONLY on validation data
# ------------------------------------------------------------

calibrator_v3 = IsotonicRegression(
    y_min=0.0,
    y_max=1.0,
    out_of_bounds="clip"
)

calibrator_v3.fit(
    val_prob_v3,
    y3v3_val
)

print("=" * 60)
print("MODEL 3 V3 — CALIBRATOR TRAINED")
print("=" * 60)

print("Calibration method: Isotonic Regression")
print("Calibration data  : 2018 validation")
print("Test data          : 2019 untouched")

MODEL 3 V3 — CALIBRATOR TRAINED
Calibration method: Isotonic Regression
Calibration data  : 2018 validation
Test data          : 2019 untouched


In [ ]:
# ============================================================
# MODEL 3 V3 — CALIBRATED PROBABILITIES
# ============================================================

val_prob_v3_calibrated = (
    calibrator_v3.predict(
        val_prob_v3
    )
)

test_prob_v3_calibrated = (
    calibrator_v3.predict(
        test_prob_v3
    )
)

print("=" * 60)
print("CALIBRATED PROBABILITY STATISTICS")
print("=" * 60)

print("\nValidation:")
print(
    pd.Series(
        val_prob_v3_calibrated
    ).describe()
)

print("\nTest:")
print(
    pd.Series(
        test_prob_v3_calibrated
    ).describe()
)

CALIBRATED PROBABILITY STATISTICS

Validation:
count    18281.000000
mean         0.015918
std          0.073520
min          0.000000
25%          0.000000
50%          0.000000
75%          0.004967
max          0.900000
dtype: float64

Test:
count    19246.000000
mean         0.014868
std          0.064623
min          0.000000
25%          0.000000
50%          0.000000
75%          0.004967
max          0.900000
dtype: float64


In [ ]:
# ============================================================
# MODEL 3 V3 — CALIBRATION COMPARISON
# ============================================================

raw_brier_test = brier_score_loss(
    y3v3_test,
    test_prob_v3
)

calibrated_brier_test = brier_score_loss(
    y3v3_test,
    test_prob_v3_calibrated
)

print("=" * 60)
print("MODEL 3 V3 — BRIER SCORE COMPARISON")
print("=" * 60)

print(
    f"Raw CatBoost Brier        : "
    f"{raw_brier_test:.6f}"
)

print(
    f"Calibrated Brier          : "
    f"{calibrated_brier_test:.6f}"
)

MODEL 3 V3 — BRIER SCORE COMPARISON
Raw CatBoost Brier        : 0.032017
Calibrated Brier          : 0.010671


In [ ]:
# ============================================================
# MODEL 3 V3 — CALIBRATION CURVE AFTER CALIBRATION
# ============================================================

from sklearn.calibration import calibration_curve

prob_true_cal, prob_pred_cal = calibration_curve(
    y3v3_test,
    test_prob_v3_calibrated,
    n_bins=10,
    strategy="quantile"
)

calibration_after_v3 = pd.DataFrame({
    "mean_predicted_probability":
        prob_pred_cal,

    "actual_positive_rate":
        prob_true_cal
})

print("=" * 60)
print("MODEL 3 V3 — CALIBRATION AFTER ISOTONIC REGRESSION")
print("=" * 60)

display(
    calibration_after_v3.round(4)
)

MODEL 3 V3 — CALIBRATION AFTER ISOTONIC REGRESSION


,mean_predicted_probability,actual_positive_rate
0,0.0000,0.0004
1,0.0013,0.0040
2,0.0038,0.0030
3,0.0065,0.0071
4,0.0159,0.0170
5,0.1300,0.1271


In [ ]:
# ============================================================
# MODEL 3 V3 — FINAL PRODUCTION EXPORT
# ============================================================

import os
import json
import joblib

MODEL_3_DIR = "/content/model_3_v3_production"

os.makedirs(
    MODEL_3_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. CatBoost model
# ------------------------------------------------------------

MODEL_3_PATH = os.path.join(
    MODEL_3_DIR,
    "model_3_v3_catboost.cbm"
)

model_3_v3.save_model(
    MODEL_3_PATH
)

# ------------------------------------------------------------
# 2. Isotonic calibrator
# ------------------------------------------------------------

CALIBRATOR_PATH = os.path.join(
    MODEL_3_DIR,
    "model_3_v3_isotonic_calibrator.joblib"
)

joblib.dump(
    calibrator_v3,
    CALIBRATOR_PATH
)

# ------------------------------------------------------------
# 3. Feature schema
# ------------------------------------------------------------

FEATURE_SCHEMA = {
    "model_name": "Model 3 V3",
    "purpose": "Zero-production risk prediction",

    "model_type": "CatBoostClassifier",

    "feature_count": len(
        MODEL_3_V3_FEATURES
    ),

    "features": MODEL_3_V3_FEATURES,

    "categorical_features": CAT_FEATURES_V3,

    "target": TARGET_V3,

    "forbidden_features": [
        "production",
        "yield",
        "current_crop_year"
    ],

    "raw_classification_threshold": 0.90,

    "calibration": {
        "method": "IsotonicRegression",
        "training_period": "2018",
        "test_period": "2019"
    },

    "metrics": {
        "roc_auc": float(v3_test_roc_auc),
        "pr_auc": float(v3_test_pr_auc),
        "precision": float(test_precision_v3),
        "recall": float(test_recall_v3),
        "f1": float(test_f1_v3),
        "raw_brier": float(raw_brier_test),
        "calibrated_brier": float(
            calibrated_brier_test
        )
    },

    "confusion_matrix": {
        "TN": int(cm_v3[0, 0]),
        "FP": int(cm_v3[0, 1]),
        "FN": int(cm_v3[1, 0]),
        "TP": int(cm_v3[1, 1])
    }
}

SCHEMA_PATH = os.path.join(
    MODEL_3_DIR,
    "model_3_v3_feature_schema.json"
)

with open(
    SCHEMA_PATH,
    "w"
) as f:

    json.dump(
        FEATURE_SCHEMA,
        f,
        indent=4
    )

print("=" * 60)
print("MODEL 3 V3 — PRODUCTION EXPORT COMPLETE")
print("=" * 60)

print("\nCatBoost:")
print(MODEL_3_PATH)

print("\nCalibrator:")
print(CALIBRATOR_PATH)

print("\nSchema:")
print(SCHEMA_PATH)

print("\nFiles:")
for file in os.listdir(MODEL_3_DIR):
    print("✓", file)

MODEL 3 V3 — PRODUCTION EXPORT COMPLETE

CatBoost:
/content/model_3_v3_production/model_3_v3_catboost.cbm

Calibrator:
/content/model_3_v3_production/model_3_v3_isotonic_calibrator.joblib

Schema:
/content/model_3_v3_production/model_3_v3_feature_schema.json

Files:
✓ model_3_v3_isotonic_calibrator.joblib
✓ model_3_v3_feature_schema.json
✓ model_3_v3_catboost.cbm


In [ ]:
import shutil

zip_path = shutil.make_archive(
    "/content/model_3_v3_production",
    "zip",
    "/content",
    "model_3_v3_production"
)

print(zip_path)

/content/model_3_v3_production.zip


In [ ]:
from google.colab import drive
drive.mount('/content/drive')